In [1]:
!pip install gradio transformers torch accelerate openai bitsandbytes sentencepiece dotenv

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   ------------------ --------------------- 5.0/10.7 MB 27.4 MB/s eta 0:00:01
   ---------------------------------------- 10.7/10.7 MB 30.2 MB/s  0:00:00
   ---------------------------------------- 0.0/616.3 kB ? eta -:--:--
   ---------------------------------------- 616.3/616.3 kB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 43.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 22.6 MB/s  0:00:00
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   -- ------------------------------------- 6.3/113.7 MB 32.2 MB/s eta 0:00:04
   ----- ---------------------------------- 14.4/113.7 MB 34.9 MB/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Sunakshi\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\torch\\include\\ATen\\native\\transformers\\cuda\\mem_eff_attention\\iterators\\predicated_tile_access_iterator_residual_last.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [2]:
import json
import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI
from dotenv import load_dotenv

In [3]:
# =========================
# 🔹 OpenRouter Setup
# =========================


#connecting to openrouter
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not OPENROUTER_API_KEY:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not OPENROUTER_API_KEY.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif OPENROUTER_API_KEY.strip() != OPENROUTER_API_KEY:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

    
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

python-dotenv could not parse statement starting at line 2


API key found and looks good so far!


In [4]:
# =========================
# 🔹 Hugging Face Setup
# =========================
HG_TOKEN = os.getenv("HG-TOKEN")

if HG_TOKEN:
    if HG_TOKEN.startswith("hf_") and HG_TOKEN.strip() == HG_TOKEN:
        print("Hugging Face token was found.")
    else:
        print("Hugging Face token format is not correct.")
else:
    print("No Hugging Face token found. Some HF models may fail.")


HF_MODELS = {
    "Phi-3 Mini": "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}


Hugging Face token was found.


In [5]:
loaded_models = {}

def load_hf_model(model_key):
    if model_key in loaded_models:
        return loaded_models[model_key]

    model_name = HF_MODELS[model_key]
    print(f"Loading Hugging Face model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HG_TOKEN
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HG_TOKEN,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    loaded_models[model_key] = (tokenizer, model)
    return tokenizer, model

In [6]:
def build_optimize_prompt(code):
    return f"""
You are a staff software engineer at a startup. You have been given lots of messy code.
You have been tasked with refactoring the following code.

Optimize the following code to:
- Improve readability
- Optimize logic
- Follow best practices

Return ONLY valid JSON in this format:

{{
  "optimized_code": "...",
  "improvements": ["improvement 1", "improvement 2"]
}}

Code:
{code}
"""

In [7]:
#building the test case generator prompt
def build_test_prompt(code):
    return f"""
You are a staff software test engineer at a startup. 


Generate pytest unit tests for the following code.
Cover all cases including:
- General cases
- Edge cases

Return ONLY valid JSON in this format:

{{
  "unit_tests": "pytest code"
}}

Code:
{code}
"""

In [8]:
def generate_with_openrouter(prompt, model_name):
    if not openrouter_client:
        raise ValueError("OpenRouter API key not configured.")

    response = openrouter_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    return response.choices[0].message.content.strip()

def generate_with_huggingface(prompt, model_key):
    tokenizer, model = load_hf_model(model_key)

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=1500,
        temperature=0.5,
        do_sample=True
    )

    generated_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()


In [9]:
def validate_json(text):
    try:
        return json.loads(text), None
    except Exception as e:
        return None, str(e)

In [12]:
def run_assistant(task_type, code, model_source, model_choice):

    if task_type == "Refactor":
        prompt = build_optimize_prompt(code)
    else:
        prompt = build_test_prompt(code)

    try:
        if model_source == "OpenRouter":
            raw_output = generate_with_openrouter(prompt, model_choice)
        else:
            raw_output = generate_with_huggingface(prompt, model_choice)

        data, error = validate_json(raw_output)

        if error:
            return f"JSON Error: {error}", raw_output

        return "Success", json.dumps(data, indent=2)

    except Exception as e:
        return f"Error: {str(e)}", None

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Code Refactoring & Test Generator Assistant")

    task_type = gr.Radio(
        ["Refactor", "Generate Tests"],
        value="Refactor",
        label="Task Type"
    )

    code_input = gr.Textbox(
        label="Paste Code Here",
        lines=15
    )

    model_source = gr.Radio(
        ["OpenRouter", "Hugging Face"],
        value="Hugging Face",
        label="Model Source"
    )

    model_choice = gr.Dropdown(
        choices=["Phi-3 Mini", "TinyLlama", "openai/gpt-4o-mini"],
        value="Phi-3 Mini",
        label="Model"
    )

    status = gr.Textbox(label="Status")
    output_box = gr.Textbox(label="Output", lines=20)

    run_btn = gr.Button("Run")

    run_btn.click(
        run_assistant,
        inputs=[task_type, code_input, model_source, model_choice],
        outputs=[status, output_box]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Loading Hugging Face model: microsoft/Phi-3-mini-4k-instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\Sunakshi\projects\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunakshi\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fal

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
import json,random,datetime

# pretend this came from somewhere
data = '[{"city":"New York","state":"NY","country":"USA","date":"2026-03-10","footfall":120,"sales":40,"revenue":4000},{"city":"Austin","state":"TX","country":"USA","date":"2026-03-10","footfall":90,"sales":30,"revenue":2700},{"city":"Seattle","state":"WA","country":"USA","date":"2026-03-10","footfall":70,"sales":25,"revenue":2100}]'

x = json.loads(data)

tot = 0
r = 0
stuff = []

for i in range(0,len(x)):
    a = x[i]
    if 'revenue' in a:
        r = r + a['revenue']
    else:
        r = r + 0

    if a.get("footfall") != None:
        if a["footfall"] > 0:
            conv = a["sales"] / a["footfall"]
        else:
            conv = 0
    else:
        conv = None

    thing = {"c":a["city"],"d":a["date"],"conv":conv,"rev":a.get("revenue")}
    stuff.append(thing)

    if random.random() > 0.7:
        print("random debug:",a)

# weird extra loop
for s in stuff:
    if s["conv"] != None:
        tot = tot + s["conv"]
    else:
        pass

avg = tot / len(stuff)

print("avg conversion??",avg)
print("total revenue maybe:",r)

# unnecessary function
def do_something(d):
    res=[]
    for i in d:
        if i["rev"] > 2000:
            res.append(i)
        else:
            if i["rev"] == None:
                print("missing rev")
            else:
                continue
    return res

big = do_something(stuff)

print("stores with big revenue",big)

# random date manipulation that doesn't really belong
today = datetime.datetime.now()
for i in range(3):
    print("checking day", today - datetime.timedelta(days=i))